# 03 — US data and two calendars

Checks for the US market (v1.1–v1.2): the 10 US stocks, the EUR/USD rate, and what happens when Lisbon and New York calendars are combined.

In [1]:
import pandas as pd
import plotly.express as px

from portfolio import fx, metrics
from portfolio.config import load_settings
from portfolio.db import get_connection, load_close_prices

settings = load_settings()
conn = get_connection(settings.database_path)
raw = load_close_prices(conn, settings.all_symbols)
conn.close()

pd.DataFrame({
    "days": raw.notna().sum(),
    "first": raw.apply(lambda s: s.first_valid_index().date()),
    "last": raw.apply(lambda s: s.last_valid_index().date()),
    "currency": [settings.currency(s) if s not in settings.fx_symbols.values() else "FX" for s in raw.columns],
})

,days,first,last,currency
AAPL,2232,2018-01-02,2026-11-17,USD
BCP.LS,2273,2018-01-02,2026-11-17,EUR
CAT,2232,2018-01-02,2026-11-17,USD
EDP.LS,2273,2018-01-02,2026-11-17,EUR
EURUSD=X,2311,2018-01-01,2026-11-17,FX
GALP.LS,2273,2018-01-02,2026-11-17,EUR
GOOGL,2232,2018-01-02,2026-11-17,USD
JMT.LS,2273,2018-01-02,2026-11-17,EUR
JNJ,2232,2018-01-02,2026-11-17,USD
JPM,2232,2018-01-02,2026-11-17,USD


## 1. EUR/USD rate — plausible range and no spikes

In [2]:
fx_symbol = settings.fx_symbols["USD"]
eurusd = raw[fx_symbol].dropna()
print(eurusd.describe().round(4))
print("Daily moves above 3%:", (eurusd.pct_change().abs() > 0.03).sum())
px.line(eurusd, title="EUR/USD (USD per 1 EUR)").show()

count    2311.0000
mean        1.1251
std         0.0548
min         0.9596
25%         1.0862
50%         1.1255
75%         1.1667
max         1.2510
Name: EURUSD=X, dtype: float64
Daily moves above 3%: 0


## 2. Days when only one of the two markets traded

In [3]:
lisbon = [s for s in settings.tickers + settings.benchmarks if settings.currency(s) == "EUR"]
new_york = [s for s in settings.tickers + settings.benchmarks if settings.currency(s) == "USD"]
lisbon_open = raw[lisbon].notna().any(axis=1)
ny_open = raw[new_york].notna().any(axis=1)

only = pd.DataFrame({
    "Only Lisbon open": (lisbon_open & ~ny_open).groupby(raw.index.year).sum(),
    "Only New York open": (~lisbon_open & ny_open).groupby(raw.index.year).sum(),
})
only

,Only Lisbon open,Only New York open
date,,
2018,7,3
2019,6,3
2020,6,2
2021,7,1
2022,7,1
2023,8,3
2024,7,3
2025,8,3
2026,6,2


## 3. After conversion to euros and alignment

In [ ]:
fx_columns = {symbol: currency for currency, symbol in settings.fx_symbols.items()}
rates = fx.to_new_york_close(raw[list(fx_columns)].rename(columns=fx_columns))
assets = raw.drop(columns=list(fx_columns))
foreign = {s: settings.currency(s) for s in assets.columns if settings.currency(s) != settings.base_currency}
aligned = metrics.align_calendars(fx.to_base_currency(assets, foreign, rates))

print("Dates kept:", len(aligned), "| observations per year:", round(metrics.periods_per_year(aligned.index), 1))
# Values carried forward because that market was closed
(assets.reindex(aligned.index).isna()).sum().rename("filled days")

## 4. Extreme daily moves in euros (|return| > 15%)

In [5]:
returns = metrics.simple_returns(aligned)
big = returns[(returns.abs() > 0.15).any(axis=1)]
big.round(3)

,AAPL,BCP.LS,CAT,EDP.LS,GALP.LS,GOOGL,JMT.LS,JNJ,JPM,KO,MCD,NEE,NVDA,NVG.LS,PSI20.LS,SON.LS,XOM,^GSPC,^NDX
date,,,,,,,,,,,,,,,,,,,
2018-11-16,0.010,-0.014,0.003,0.002,0.010,-0.004,0.012,0.009,-0.002,0.007,0.021,0.006,-0.188,-0.009,-0.001,0.006,0.009,0.001,-0.005
2020-03-09,-0.092,-0.152,-0.155,-0.072,-0.165,-0.075,-0.055,-0.053,-0.148,-0.075,-0.074,-0.052,-0.091,-0.095,-0.087,-0.089,-0.135,-0.089,-0.082
2020-03-13,0.127,-0.007,0.087,0.022,0.014,0.100,0.012,0.078,0.188,0.044,0.048,0.068,0.121,-0.015,0.008,-0.010,0.032,0.100,0.108
2020-03-16,-0.126,-0.096,-0.060,-0.044,-0.036,-0.114,0.020,-0.051,-0.147,-0.064,-0.157,-0.089,-0.182,-0.060,-0.044,-0.053,-0.093,-0.118,-0.120
2020-03-24,0.093,0.070,0.096,0.099,0.132,0.065,0.103,0.065,0.112,0.043,0.174,0.129,0.164,0.097,0.078,0.078,0.120,0.087,0.071
2020-11-09,-0.026,0.189,0.052,0.011,0.174,-0.005,0.009,0.020,0.128,0.056,-0.022,-0.009,-0.070,0.055,0.045,0.043,0.119,0.005,-0.028
2022-11-10,0.096,-0.007,0.037,0.024,-0.005,0.082,-0.010,0.018,0.048,0.042,-0.001,0.070,0.150,-0.011,0.008,-0.002,0.021,0.062,0.081
2023-05-25,0.008,-0.006,0.002,-0.017,-0.021,0.023,-0.008,-0.013,0.004,-0.007,-0.000,-0.010,0.245,0.000,-0.011,0.002,-0.017,0.010,0.026
2024-02-22,0.011,0.002,0.015,-0.014,-0.009,0.010,-0.015,0.010,0.011,-0.002,0.006,-0.008,0.163,-0.013,-0.008,-0.011,-0.002,0.020,0.029


## 5. FX timing — Yahoo's rate vs. the New York close

Yahoo's daily EUR/USD value for day D is taken at the start of the London day, so it misses the currency moves of D.
On 10 Nov 2022 (US inflation surprise, after 13:30 UTC) the dollar fell about 2% and the S&P 500 rose 5.5%:
if the rate were right, Nvidia and the S&P 500 should gain **less** in euros than in dollars that day.

In [ ]:
yahoo = raw[[fx_symbol]].rename(columns={fx_symbol: "USD"})
fixed = fx.to_new_york_close(yahoo)
check = ["NVDA", "^GSPC"]

def returns_in_eur(rates):
    return metrics.simple_returns(fx.to_base_currency(raw[check].dropna(), {s: "USD" for s in check}, rates))

day = "2022-11-10"
pd.DataFrame({
    "in USD": metrics.simple_returns(raw[check].dropna()).loc[day],
    "in EUR, Yahoo rate": returns_in_eur(yahoo).loc[day],
    "in EUR, New York close": returns_in_eur(fixed).loc[day],
}).round(4)

In [ ]:
# Over the whole period: annualised volatility of the S&P 500 in each version
pd.Series({
    "in USD": metrics.simple_returns(raw["^GSPC"].dropna()).std() * 252 ** 0.5,
    "in EUR, Yahoo rate": returns_in_eur(yahoo)["^GSPC"].std() * 252 ** 0.5,
    "in EUR, New York close": returns_in_eur(fixed)["^GSPC"].std() * 252 ** 0.5,
}).round(4)

## Conclusions

- **EUR/USD:** plausible throughout — between 0.96 (below parity, 2022) and 1.25, with no daily move above 3% (largest +2.8%, April 2025 tariff turmoil). No data errors.
- **Calendars:** within each market there are no gaps: the US series have 0 missing days (251 observations a year) and Portugal only the 12 PSI days already found in notebook 01. Mixing both markets would leave 6–8 days a year with only Lisbon open and 1–3 with only New York open, which supports showing one market at a time (v1.2).
- **Extreme moves:** 8 days above 15%, all real events — Nvidia's weak results (Nov 2018), the COVID crash and rebound (13, 16 and 24 Mar 2020), Nvidia's AI-driven results (May 2023, Feb 2024), the DeepSeek shock (Jan 2025) and the US tariff pause (9 Apr 2025). They are kept. Nvidia's stock splits (2021, 2024) do not show up, so adjusted prices work as expected.
- **FX timing:** Yahoo's daily EUR/USD close is taken at the start of the London day, not at the New York close. On 10 Nov 2022 (US inflation surprise) the dollar fell about 2% after 13:30 UTC, but the move only appears in Yahoo's 11 Nov value. Long-run returns are unaffected, but daily figures in euros would carry a one-day mismatch between stock and currency moves. The dashboard fixes this with `fx.to_new_york_close`, which uses Yahoo's next-day value as the rate for each New York close. After the fix, on 10 Nov 2022 Nvidia gains 12.4% in euros against 14.3% in dollars (it was 15.0% with Yahoo's rate), and the S&P 500's volatility in euros drops from 20.2% to 19.3% a year, close to the 19.1% in dollars.